# Tiny GPT From Scratch on Jetson Orin Nano 8GB (JetPack 6)

## Overview

In this notebook, we build and train a tiny GPT-style decoder-only language model from scratch using PyTorch, on a character-level dataset (Tiny Shakespeare).

What "from scratch" means here:
- We implement tokenization (character vocabulary) ourselves.
- We implement causal self-attention, masking, transformer blocks, and generation.
- We train with a standard next-token prediction objective.

What to expect on Tiny Shakespeare:
- Early outputs are mostly noise.
- After enough steps, generated text starts to resemble Shakespeare-like structure and spelling patterns.
- This is still a very small model and dataset, so quality is limited.

In [1]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
    print("set_float32_matmul_precision('high') enabled")

torch version: 2.9.1
cuda available: True
cuda device: Orin
using device: cuda
set_float32_matmul_precision('high') enabled


**Jetson monitoring tip**

In a separate terminal, run `tegrastats` during training to watch GPU/CPU memory, utilization, and thermals.

In [2]:
import os
import subprocess
from pathlib import Path

DATA_PATH = Path("input.txt")
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not DATA_PATH.exists():
    print("input.txt not found. Downloading Tiny Shakespeare...")
    try:
        subprocess.run(["wget", "-O", str(DATA_PATH), URL], check=True)
        print("Downloaded with wget.")
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("wget unavailable or failed. Falling back to urllib...")
        import urllib.request
        urllib.request.urlretrieve(URL, str(DATA_PATH))
        print("Downloaded with urllib.")
else:
    print("Found existing input.txt")

size_bytes = DATA_PATH.stat().st_size
print(f"file size: {size_bytes:,} bytes")

text_preview = DATA_PATH.read_text(encoding="utf-8")[:300]
print("\nPreview (first ~300 chars):\n")
print(text_preview)

Found existing input.txt
file size: 1,115,394 bytes

Preview (first ~300 chars):

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## Tokenizer (Character-Level)

We build a vocabulary of unique characters, then create:
- `stoi`: string character to integer id
- `itos`: integer id to character
- `encode`/`decode` helpers

Then we encode the entire dataset into a `torch.long` tensor and split into train/validation sets.

In [3]:
import torch
from pathlib import Path

text = Path("input.txt").read_text(encoding="utf-8")
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("vocab size:", vocab_size)
print("dataset length:", len(data))
print("train length:", len(train_data))
print("val length:", len(val_data))

vocab size: 65
dataset length: 1115394
train length: 1003854
val length: 111540


## Batch Sampler

For next-token prediction:
- `x` is a chunk of tokens of shape `(B, T)`
- `y` is the same chunk shifted by one token, also `(B, T)`

So each position in `x` predicts the corresponding next token in `y`.

In [4]:
# Jetson-constrained defaults (Orin Nano 8GB)
block_size = 256
n_layers = 4
n_heads = 4
d_model = 256
dropout = 0.1

batch_size = 16
grad_accum_steps = 4

max_steps = 6000
eval_every = 500
eval_iters = 100

learning_rate = 3e-4
min_lr = 3e-5
warmup_steps = 300
weight_decay = 0.1
beta1, beta2 = 0.9, 0.95
grad_clip = 1.0

amp_enabled = (device == "cuda")


def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x shape:", tuple(xb.shape), "y shape:", tuple(yb.shape))

x shape: (16, 256) y shape: (16, 256)


## Model Components

We implement a minimal GPT decoder with:
- **CausalSelfAttention**: one projection for qkv, split into heads, scaled dot-product attention
- **Causal mask**: lower-triangular mask (registered buffer) so tokens cannot attend to future tokens
- **MLP**: 4x hidden expansion with GELU and dropout
- **Block**: pre-layernorm + residual connections
- **GPT**: token + positional embeddings, stacked blocks, final norm, vocab projection

`forward()` returns logits and optional cross-entropy loss. `generate()` supports temperature and optional top-k sampling.

In [5]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.c_attn = nn.Linear(d_model, 3 * d_model)
        self.c_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.fc = nn.Linear(d_model, 4 * d_model)
        self.proj = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layers: int
    n_heads: int
    d_model: int
    dropout: float


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.block_size, config.d_model)
        self.drop = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([
            Block(config.d_model, config.n_heads, config.block_size, config.dropout)
            for _ in range(config.n_layers)
        ])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.config.block_size:
            raise ValueError(f"sequence length {T} > block_size {self.config.block_size}")

        pos = torch.arange(0, T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        x = self.drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, self.config.vocab_size),
                targets.view(B * T)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [6]:
config = GPTConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layers=n_layers,
    n_heads=n_heads,
    d_model=d_model,
    dropout=dropout,
)

model = GPT(config).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {num_params / 1e6:.2f}M")

dummy_x = torch.randint(0, vocab_size, (2, 16), device=device)
logits, loss = model(dummy_x, dummy_x)
print("logits shape:", tuple(logits.shape))
print("loss shape:", tuple(loss.shape) if hasattr(loss, "shape") else "scalar")
print("loss is scalar:", loss.ndim == 0)

parameters: 3.26M
logits shape: (2, 16, 65)
loss shape: ()
loss is scalar: True


## Training Utilities

We add two helpers:
- `estimate_loss(model)`: evaluates average train/val loss over a few mini-batches without gradients.
- `get_lr(step)`: warmup + cosine decay schedule down to `min_lr`.

In [7]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def get_lr(step):
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    if step > max_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / max(1, (max_steps - warmup_steps))
    decay_ratio = min(max(decay_ratio, 0.0), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

## Training Loop

Training uses:
- AdamW optimizer
- FP16 AMP + GradScaler (enabled only on CUDA)
- Gradient accumulation (`grad_accum_steps`)
- Gradient clipping (`max_norm=1.0`)

Every `eval_every` steps we print:
- step, learning rate, train loss, val loss
- a short generated sample (~300 chars)

In [8]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(beta1, beta2),
    weight_decay=weight_decay,
)

scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

model.train()

for step in range(max_steps + 1):
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)

    for _ in range(grad_accum_steps):
        xb, yb = get_batch("train")
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            _, loss = model(xb, yb)
            loss = loss / grad_accum_steps

        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

    if step % eval_every == 0:
        losses = estimate_loss(model)
        print(f"step {step:5d} | lr {lr:.6f} | train {losses['train']:.4f} | val {losses['val']:.4f}")

        context = torch.zeros((1, 1), dtype=torch.long, device=device)
        sample_ids = model.generate(context, max_new_tokens=300, temperature=0.9, top_k=50)[0].tolist()
        print("sample:\n" + decode(sample_ids))
        print("-" * 80)

/tmp/ipykernel_191705/1407356680.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


step     0 | lr 0.000001 | train 4.2066 | val 4.2054
sample:

RdCrDK
F;HPREf?AWsSYTkbuv&lS.PElHxlWGO;.dd:tc3F,,AiyW?fuZOhBe DdyHIaMfNRpzsrorUsLF,xc?CszOwrepIFLfXFYI?hgHN3WDIJtBP$jiwAoO;yD
BkKKhyJ
YX$KBI?gg NyaalVWV3mmDEI'foOTUuLSN?LqHsFuRtROTrRODoqsc
DxjyoqRxPTk!lyndea
Oz oABDzeCtS:scY&.fI.!$-:Vt& OcKfN
A:TMLLabxgE&P3zMuIDo'YIaVD?bh;P$qEhO?etXnVo!$jfAq hT;Ltyv
--------------------------------------------------------------------------------
step   500 | lr 0.000299 | train 2.1328 | val 2.1818
sample:

VARDIZY E:
He vil, dend and thit.

Nenl VI Gon, dul wherd; he sthis bove sus sengennd bes
Thish pave of 'parets my ores,
Sale bult nof nlithe bit cthat thoust doute pa thend thange,
At irot ucror, bullbuth the yerrare ad meat ant.

YUKEROLE :
Whaprith thal me youngoust?


FVINGLIO:
Bule, heis sponch
--------------------------------------------------------------------------------
step  1000 | lr 0.000290 | train 1.6846 | val 1.8477
sample:

Bone crotce
To dard deaus the what hath as by wand

In [9]:
ckpt = {
    "model_state": model.state_dict(),
    "config": {
        "vocab_size": vocab_size,
        "block_size": block_size,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "d_model": d_model,
        "dropout": dropout,
        "chars": chars,
    },
}

torch.save(ckpt, "gpt_char_ckpt.pt")
print("saved checkpoint to gpt_char_ckpt.pt")

saved checkpoint to gpt_char_ckpt.pt


## Load Checkpoint + Generate

We now create a fresh model instance, load the saved weights/config, and generate a longer sample.

In [10]:
loaded = torch.load("gpt_char_ckpt.pt", map_location=device)
cfg_dict = loaded["config"]

loaded_config = GPTConfig(
    vocab_size=cfg_dict["vocab_size"],
    block_size=cfg_dict["block_size"],
    n_layers=cfg_dict["n_layers"],
    n_heads=cfg_dict["n_heads"],
    d_model=cfg_dict["d_model"],
    dropout=cfg_dict["dropout"],
)

model_loaded = GPT(loaded_config).to(device)
model_loaded.load_state_dict(loaded["model_state"])
model_loaded.eval()

chars_loaded = cfg_dict["chars"]
itos_loaded = {i: ch for i, ch in enumerate(chars_loaded)}

def decode_loaded(ids):
    return "".join(itos_loaded[i] for i in ids)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
with torch.no_grad():
    out = model_loaded.generate(context, max_new_tokens=600, temperature=0.9, top_k=50)

print(decode_loaded(out[0].tolist()))


Show the queen of his party friends
Within the child other grave. Come, come, let me not?

LEONTES:
We see urged some fair, that she hath done lies
'Twixt them, and thou repose the pride?' O, how may
That poison'd us by no horse.

POLIXENES:
No, no so so.

LEONTES:
He cannot with you.

CAMILLO:
O, sir; I am a respected borough in this mother
To join: and as I speak, the gods and side
That standers from me from from a dull bear to straw
What did reply the complexion of sick fair?

First Lord:
What a man near, that news shall devil,
Thereby lives to make his presently seat.

LORD WILLOUGHBY:
O, 


## Next Steps

- Character-level models are simple but inefficient and limited in quality for larger-scale language modeling.
- A strong next upgrade is using a subword tokenizer (BPE/SentencePiece) and training a token-level GPT.
- "Instruction tuning" is a separate phase done after base next-token pretraining.

## OOM Safety (Jetson 8GB)

If you hit out-of-memory errors, reduce memory in this order:
1. Lower `batch_size` (e.g., 16 -> 8 -> 4)
2. Lower `block_size` (e.g., 256 -> 192 -> 128)
3. Lower `d_model` (e.g., 256 -> 192)

You can also reduce `n_layers` or increase `grad_accum_steps` to preserve effective batch size.

Run this notebook with:

`jupyter notebook tiny_gpt_from_scratch_jetson.ipynb`

If you use JupyterLab:

`jupyter lab tiny_gpt_from_scratch_jetson.ipynb`

# =========================
# ONE-CELL: Load checkpoint + rebuild model + interactive chat loop (Char-level Tiny GPT)
# Assumptions:
#  - You already trained once and have: ./gpt_char_ckpt.pt
#  - This cell is self-contained (no need to run earlier cells)
#  - It will start an interactive prompt using Python input()
# =========================

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------- Settings ----------
CKPT_PATH = "gpt_char_ckpt.pt"
TEMPERATURE = 0.9
TOP_K = 50
MAX_NEW_TOKENS = 300

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("Torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)

# ---------- Load checkpoint ----------
ckpt = torch.load(CKPT_PATH, map_location=device)
cfg = ckpt["config"]

# ---------- Rebuild tokenizer (char-level) ----------
chars = cfg["chars"]
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
vocab_size = cfg["vocab_size"]

def encode(s: str):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print(f"Loaded ckpt: {CKPT_PATH}")
print(f"Vocab size: {vocab_size} | block_size: {cfg['block_size']} | d_model: {cfg['d_model']} | layers: {cfg['n_layers']} | heads: {cfg['n_heads']}")
print("Note: base model = text completion. Prompt style like 'User: ...\nAssistant:' works better.")

# ---------- Model definition (matches training notebook) ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.c_attn = nn.Linear(d_model, 3 * d_model)
        self.c_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size),
        )

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.fc = nn.Linear(d_model, 4 * d_model)
        self.proj = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(d_model, n_heads, block_size, dropout) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"sequence length {T} > block_size {self.block_size}")

        pos = torch.arange(0, T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        x = self.drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, logits.size(-1)), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)

        return idx

# ---------- Build model + load weights ----------
model = GPT(
    vocab_size=cfg["vocab_size"],
    d_model=cfg["d_model"],
    n_layers=cfg["n_layers"],
    n_heads=cfg["n_heads"],
    block_size=cfg["block_size"],
    dropout=cfg["dropout"],
).to(device)

model.load_state_dict(ckpt["model_state"])
model.eval()

param_m = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model params: {param_m:.2f}M")

@torch.no_grad()
def complete(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_k=TOP_K):
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(out[0].tolist())


def chat_loop():
    print("\nChat started. Type 'exit' to stop.")
    while True:
        try:
            user_text = input("You: ").strip()
        except EOFError:
            print("\nInput stream closed; exiting.")
            break

        if user_text.lower() in {"exit", "quit"}:
            print("Bye.")
            break
        if not user_text:
            continue

        prompt = f"User: {user_text}\nAssistant:"
        try:
            out = complete(prompt, max_new_tokens=220)
            print("Assistant:", out[len(prompt):].strip(), "\n")
        except KeyError as e:
            print(f"Character not in vocab: {e}. Use simpler text or retrain with broader data.\n")

chat_loop()

Torch: 2.9.1
MPS available: False
CUDA available: True
GPU: Orin
Device: cuda
Loaded ckpt: gpt_char_ckpt.pt
Vocab size: 65 | block_size: 256 | d_model: 256 | layers: 4 | heads: 4
Note: base model = text completion. Prompt style like 'User: ...
Assistant:' works better.
Model params: 3.26M

Chat started. Type 'exit' to stop.
Assistant: The fields up in this love, nor with this incense?
By us reconcilence the royal rest in the old of cast
The old friends of this intent face, as they have
Here precipling here accomplished laid.
'Twas the way, if they wi 

Assistant: thou art art in this wert hands.

Lord:
And many in my cousin or death?

EXETER:
Then ever have no more petition me,
Stand that I do not put up, and sell our good soul
The walls bite that cannot we stand upon my mother' 

Bye.
